# ai-analyst-eval on Kaggle

Runs the same `aae` package as a laptop, inside a Kaggle notebook: small models through Ollama on the Kaggle GPU, frontier models through the Kaggle model proxy or your own API keys.

**Before you run**

1. Edit the first code cell. Set `REPO_URL` to the repository, `RUN_MODE` to one of `shard`, `frontier`, `local`, `all` or `report`, and for `shard` set `SCHEDULE_START` to the UTC date of the first scheduled run.
2. Settings: Internet On. Accelerator GPU T4 x2 or P100 for `local` and `all`; none for `shard`, `frontier` and `report`.
3. Frontier route. Start the notebook from the Kaggle benchmark-task page to receive the model-proxy credential (`MODEL_PROXY_URL`, `MODEL_PROXY_API_KEY`). To use the vendors' APIs instead, add `ANTHROPIC_API_KEY`, `GEMINI_API_KEY` and `OPENAI_API_KEY` under Add-ons, Secrets, or set `AAE_ROUTE=direct`. Secrets are read from the environment and never printed.
4. Optional: add `KAGGLE_USERNAME` and `KAGGLE_KEY` as Secrets so each run saves its new records to a private dataset (`aae-shards`) that the next run reads back.
5. Saved state. Attach the dataset or notebook output that holds `artifacts/`, `runs/` and `contexts/` from an earlier session as an input. The restore cell merges every saved session by run id, so nothing is counted twice. A first run with no saved state builds everything from BIRD Mini-Dev.
6. Save Version, then Save & Run All. To schedule, use Schedule on the saved version.

**Modes**

| `RUN_MODE` | What it does |
| --- | --- |
| `shard` | runs one independent slice of the remaining frontier questions; the date picks the slice |
| `frontier` | runs the main grid on F1, F2 and F3 |
| `local` | runs the main grid on M1 and M2 |
| `all` | runs both |
| `report` | skips the run and rebuilds the table, the headline numbers and the charts |

**Outputs.** New records are in `shard_out/` and in `runs/`. The results table is `results/runs_table.csv` and the charts are in `docs/`. Raw run logs contain model-generated SQL: do not publish them.

In [ ]:
REPO_URL = "https://github.com/altestxyz/ai-analyst-eval.git"
RUN_MODE = "shard"
SCHEDULE_START = ""
SHARD = "auto"
SHARD_QUESTIONS = 40
FRONTIER_REPEATS = 1
LOCAL_MODELS = ["M1", "M2"]
PUBLISH = True
RESCORE = False

In [ ]:
import os
import subprocess
import sys
import tempfile
import time


def sh(*args, check=True):
    print("$", " ".join(args), flush=True)
    return subprocess.run(list(args), check=check)


def load_secrets():
    try:
        from kaggle_secrets import UserSecretsClient
    except ImportError:
        return
    client = UserSecretsClient()
    names = ("MODEL_PROXY_URL", "MODEL_PROXY_API_KEY", "ANTHROPIC_API_KEY", "GEMINI_API_KEY", "OPENAI_API_KEY", "KAGGLE_USERNAME", "KAGGLE_KEY")
    for name in names:
        if name in os.environ:
            continue
        try:
            os.environ[name] = client.get_secret(name)
        except Exception:
            pass


load_secrets()
if RUN_MODE == "shard" and not SCHEDULE_START:
    raise SystemExit("Set SCHEDULE_START to the UTC date of the first scheduled run.")
if RUN_MODE not in ("shard", "frontier", "local", "all", "report"):
    raise SystemExit(f"Unknown RUN_MODE {RUN_MODE!r}.")
if "<owner>" in REPO_URL:
    raise SystemExit("Set REPO_URL in the first code cell to the clone URL of the repository.")

In [ ]:
if not os.path.isdir("ai-analyst-eval"):
    sh("git", "clone", "--depth", "1", REPO_URL, "ai-analyst-eval")
os.chdir("ai-analyst-eval")
sh(sys.executable, "-m", "pip", "install", "-q", "-e", ".")

In [ ]:
if PUBLISH and RUN_MODE != "report":
    sh(sys.executable, "-m", "aae.publish", "pull", check=False)
bases = ["--from", "/kaggle/input", "--from", os.path.join(tempfile.gettempdir(), "aae_pub")]
sh(sys.executable, "-m", "aae", "restore", *bases, "--to", ".")

In [ ]:
if RUN_MODE in ("local", "all"):
    sh("bash", "-c", "apt-get update -qq && apt-get install -y -qq zstd pciutils lshw > /dev/null 2>&1; curl -fsSL https://ollama.com/install.sh | sh", check=False)
    subprocess.Popen(["ollama", "serve"], stdout=open("ollama.log", "w"), stderr=subprocess.STDOUT)
    time.sleep(8)
    from aae.models import load_models

    models = load_models()
    for tag in sorted({models[k]["id"] for k in LOCAL_MODELS}):
        sh("ollama", "pull", tag)

In [ ]:
TARGETS = {"shard": "frontier", "frontier": "frontier", "local": "laptop", "all": "all"}
if RUN_MODE != "report":
    sh(sys.executable, "-m", "aae", "doctor", "--target", TARGETS[RUN_MODE])

In [ ]:
if RUN_MODE != "report" and not os.path.exists("artifacts/questions.jsonl"):
    sh(sys.executable, "-m", "aae", "setup")
    sh(sys.executable, "-m", "aae", "taxi")
    sh(sys.executable, "-m", "aae", "contexts")
    sh(sys.executable, "-m", "aae", "freeze", "write")

In [ ]:
if RUN_MODE == "shard":
    sh(sys.executable, "-m", "aae", "shard", "--schedule-start", SCHEDULE_START, "--slice", str(SHARD),
       "--slice-questions", str(SHARD_QUESTIONS), "--frontier-repeats", str(FRONTIER_REPEATS))
elif RUN_MODE in ("frontier", "local", "all"):
    extra = ["--frontier-repeats", str(FRONTIER_REPEATS)] if RUN_MODE != "local" else []
    sh(sys.executable, "-m", "aae", "run", "--target", TARGETS[RUN_MODE], *extra)

In [ ]:
if PUBLISH and RUN_MODE != "report":
    sh(sys.executable, "-m", "aae.publish", "push", check=False)

In [ ]:
if RESCORE:
    sh(sys.executable, "-m", "aae", "rescore", "run", "--runs", "runs/runs.jsonl", "--out", "results/rescored.json", "--summary", "results/rescore_summary.json")
table = ["--runs", "runs/runs.jsonl", "--out", "results/runs_table.csv"]
if RESCORE:
    table += ["--rescored", "results/rescored.json"]
os.makedirs("results", exist_ok=True)
sh(sys.executable, "-m", "aae", "table", *table)
sh(sys.executable, "-m", "aae", "headline", "--table", "results/runs_table.csv")
sh(sys.executable, "-m", "aae", "figures", "--table", "results/runs_table.csv", "--out", "docs")